##Analysis of Cursor Movement

#Creation of Data Frame for Basic reaching and Continuous tracking

#Basic Reaching

#Continuous Tracing

In [9]:
import pandas as pd
from pathlib import Path

# 1. Den Hauptordner definieren (das 'r' vor dem String ist wichtig für Windows-Pfade, damit Backslashes nicht als Steuerzeichen gelesen werden)
basis_ordner = Path("data/Continuous_tracking")

# Eine leere Liste, in der wir alle einzelnen Datensätze sammeln
alle_dfs = []

# 2. Automatisch alle CSV-Dateien suchen
# .rglob("*.csv") sucht rekursiv (also im Hauptordner UND in allen Unterordnern) nach CSV-Dateien
for datei_pfad in basis_ordner.rglob("*.csv"):
    
    # 3. CSV einlesen
    df_temp = pd.read_csv(datei_pfad)
    
    # WICHTIG: Probanden-ID aus dem Dateipfad extrahieren und als Spalte anfügen.
    # datei_pfad.parent.name gibt uns genau den Namen des Unterordners (z.B. 'ch51' oder 'ir31')
    df_temp['Participant'] = datei_pfad.parent.name
    
    # Den markierten DataFrame in unsere Liste packen
    alle_dfs.append(df_temp)

# 4. Alle Datensätze aus der Liste zu einem großen Master-DataFrame zusammenführen
# ignore_index=True sorgt dafür, dass die Zeilen von 0 bis X durchgehend neu nummeriert werden
df_tracking = pd.concat(alle_dfs, ignore_index=True)

# 5. Daten inspizieren
print(df_tracking.head())
print("\n" + "="*50 + "\n") # Nur ein optischer Trennstrich
print(df_tracking.info())

   target_width  cursor-width  target_pos_x  target_pos_y  cursor_pos_x  \
0           0.1           0.0           0.0           0.0      0.239135   
1           0.1           0.0           0.0           0.0      0.239135   
2           0.1           0.0           0.0           0.0      0.211002   
3           0.1           0.0           0.0           0.0      0.196935   
4           0.1           0.0           0.0           0.0      0.182868   

   cursor_pos_y  target_vel_x  target_vel_y  timestamp  trial_number  \
0      0.014067           0.0           0.0   0.000008             0   
1      0.014067           0.0           0.0   0.006560             0   
2      0.000000           0.0           0.0   0.012574             0   
3     -0.014067           0.0           0.0   0.019620             0   
4     -0.028134           0.0           0.0   0.027063             0   

   training  gaze_pos_x  gaze_pos_y  tracker_time  state_marker Participant  
0      True   33.900024   11.900024   

##BAsic Reaching

In [29]:
import pandas as pd
from pathlib import Path

basis_ordner = Path("data/Basic_reaching")
alle_dfs = []

# Unser Lexikon (Mapping) aus der fparams_copy.py
# Wenn Block 1 ist -> 0.2, wenn 2 -> 0.3, usw.
blob_mapping = {
    1: 0.2,
    2: 0.3,
    3: 0.4,
    4: 0.5
}

for datei_pfad in basis_ordner.rglob("*.csv"):
    df_temp = pd.read_csv(datei_pfad)
    
    # 1. Probanden-ID
    df_temp['Participant'] = datei_pfad.parent.name
    
    # 2. Ist es Training?
    df_temp['is_training'] = 'block_0' in datei_pfad.name
    
    # ===============================================================
    # NEU: 3. Blocknummer aus dem Dateinamen extrahieren
    # Wir schneiden den Text am Wort 'block_' durch, nehmen den rechten Teil,
    # schneiden ihn am nächsten '_' wieder durch und nehmen die Zahl.
    block_string = datei_pfad.name.split('block_')[1].split('_')[0]
    block_nummer = int(block_string)
    df_temp['block'] = block_nummer
    
    # NEU: 4. Blob Width zuordnen (Training [Block 0] bekommt NaN oder 0)
    # .get(block_nummer, None) sagt: Such die Nummer im Lexikon, falls nicht da -> None
    df_temp['blob_width'] = blob_mapping.get(block_nummer, None)
    # ===============================================================
    
    alle_dfs.append(df_temp)

# Zusammenführen
df_reaching = pd.concat(alle_dfs, ignore_index=True)

# Training filtern
df_reaching_clean = df_reaching[df_reaching['is_training'] == False]

# Kurzer Check, ob unsere Rekonstruktion geklappt hat!
print("Gefundene Blob Widths nach Block:")
print(df_reaching_clean.groupby('block')['blob_width'].unique())
print(df_reaching_clean.info())

Gefundene Blob Widths nach Block:
block
-1    [nan]
 1    [0.2]
 2    [0.3]
 3    [0.4]
Name: blob_width, dtype: object
<class 'pandas.core.frame.DataFrame'>
Index: 181697 entries, 0 to 243742
Data columns (total 17 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   trial           181697 non-null  int64  
 1   time            181697 non-null  float64
 2   cursor_x_pix    181697 non-null  float64
 3   cursor_y_pix    181697 non-null  float64
 4   cursor_x_cm     181697 non-null  float64
 5   cursor_y_cm     181697 non-null  float64
 6   gaze_x          181697 non-null  float64
 7   gaze_y          181697 non-null  float64
 8   target_x        181697 non-null  float64
 9   target_y        181697 non-null  float64
 10  state_marker    181697 non-null  int64  
 11  button_pressed  181697 non-null  bool   
 12  start_time      181697 non-null  float64
 13  Participant     181697 non-null  object 
 14  is_training     181697 non-null  

C:\Users\march\AppData\Local\Temp\ipykernel_34168\2069270212.py:41: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_reaching = pd.concat(alle_dfs, ignore_index=True)


## Analysis

In [8]:
print(df_tracking['Participant'].unique())
print(df_reaching['Participant'].unique())

['ch51' 'ir31' 'lj25' 'pf13']
['CH51' 'IR31' 'LJ25' 'PF13']


## Cleaning data of training trials

Cleaning continous tracking

In [10]:
# Wir filtern den Master-DataFrame und speichern das Ergebnis in einer neuen Variable
# Wichtig: Bei Booleans (True/False) schreiben wir False ohne Anführungszeichen!
df_continuous_clean = df_tracking[df_tracking['training'] == False]

# Ein kurzer Check, um sicherzugehen, dass das Filtern geklappt hat:
# .unique() sollte uns jetzt nur noch [False] ausgeben.
print("Verbleibende Werte in der 'training' Spalte:")
print(df_continuous_clean['training'].unique())

# Optional: Schauen wir, wie viele Zeilen wir vorher und nachher haben
print(f"Zeilen vorher: {len(df_tracking)}")
print(f"Zeilen nachher: {len(df_continuous_clean)}")

Verbleibende Werte in der 'training' Spalte:
[False]
Zeilen vorher: 491512
Zeilen nachher: 472480


Cleaning basic reaching

In [ ]:
# Eigentlich nicht mehr nötuig, da es von Code weiter oben übernommen wird
# Wichtig: Bei Booleans (True/False) schreiben wir False ohne Anführungszeichen!
df_reaching_clean = df_reaching[df_reaching['is_training'] == False]

# Ein kurzer Check, um sicherzugehen, dass das Filtern geklappt hat:
# .unique() sollte uns jetzt nur noch [False] ausgeben.
print("Verbleibende Werte in der 'training' Spalte:")
print(df_reaching_clean['is_training'].unique())

# Optional: Schauen wir, wie viele Zeilen wir vorher und nachher haben
print(f"Zeilen vorher: {len(df_reaching)}")
print(f"Zeilen nachher: {len(df_reaching_clean)}")

Verbleibende Werte in der 'training' Spalte:
[False]
Zeilen vorher: 243743
Zeilen nachher: 181697


# Velocity

# Basic reaching

In [ ]:
import numpy as np

# 1. Zur Sicherheit: Daten chronologisch ordnen
df_reaching_clean = df_reaching_clean.sort_values(by=['Participant', 'trial', 'time'])

# 2. Differenzen berechnen (grouped by Participant und Trial!)
df_reaching_clean['dt'] = df_reaching_clean.groupby(['Participant', 'trial'])['time'].diff()
df_reaching_clean['dx'] = df_reaching_clean.groupby(['Participant', 'trial'])['cursor_x_cm'].diff()
df_reaching_clean['dy'] = df_reaching_clean.groupby(['Participant', 'trial'])['cursor_y_cm'].diff()

# 3. Pythagoras für die absolute Distanz (in cm)
df_reaching_clean['dist_cm'] = np.sqrt(df_reaching_clean['dx']**2 + df_reaching_clean['dy']**2)

# 4. Geschwindigkeit (cm / s)
df_reaching_clean['cursor_velocity'] = df_reaching_clean['dist_cm'] / df_reaching_clean['dt']

# Kurzer Check
print(df_reaching_clean[['time', 'cursor_x_cm', 'cursor_velocity']].head())
#----------------------------------------------------------------------------------------
#Calculating mean
#----------------------------------------------------------------------------------------
# HIER ANPASSEN: Ersetze 'deine_blob_spalte' durch den echten Namen der Spalte im Reaching-Datensatz
spalte_blob_reaching = 'blob_width' 

# Gruppieren und Mittelwert berechnen
df_reaching_mean_vel = df_reaching_clean.groupby(['Participant', spalte_blob_reaching])['cursor_velocity'].mean().reset_index()

# Die Spalte zur Klarheit umbenennen
df_reaching_mean_vel = df_reaching_mean_vel.rename(columns={'cursor_velocity': 'mean_velocity_cm_s'})

print("--- Basic Reaching: Mean Velocity ---")
print(df_reaching_mean_vel.head(10))


       time  cursor_x_cm  cursor_velocity
0  3.500525     1.119375              NaN
1  3.527586     1.119375              0.0
2  3.534337     1.119375              0.0
3  3.541171     1.119375              0.0
4  3.548086     1.119375              0.0
--- Basic Reaching: Mean Velocity ---
  Participant  blob_width  mean_velocity_cm_s
0        CH51         0.2            7.838088
1        CH51         0.3            8.311523
2        CH51         0.4            8.982089
3        IR31         0.2            5.562514
4        IR31         0.3            6.036498
5        IR31         0.4            5.862652
6        LJ25         0.2            5.173713
7        LJ25         0.3            5.995182
8        LJ25         0.4            5.709873
9        PF13         0.2            5.426338
<class 'pandas.core.frame.DataFrame'>
Index: 181697 entries, 0 to 240088
Data columns (total 22 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0  

# Velocity curves

Ich wollte hier zuerst die reine velocity kurve machen, dann ist mir aber aufgefallen, dass der rückweg teilweise schneller ist als das ziel ansteuern, evt interessant falls wir sonst nichts cooles finden

In [60]:
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display

def show_vc(ausgewaehlter_proband, ausgewaehlter_block, ausgewaehltes_trial):
    # Das hier ist dein korrekt gefilterter Datensatz
    df_gefiltert = df_reaching_clean[
        (df_reaching_clean['Participant'] == ausgewaehlter_proband) & 
        (df_reaching_clean['block'] == ausgewaehlter_block) &
        (df_reaching_clean['trial'] == ausgewaehltes_trial)
    ]
    
    if df_gefiltert.empty:
        print('Keine Daten gefunden')
        return
    

    plt.figure(figsize=(10, 5))
    sns.scatterplot(data=df_gefiltert, x='time', y='cursor_velocity', hue='state_marker', palette='bright')
    plt.title("Velocity nach State-Marker")
    plt.show()
    
    plt.figure(figsize=(5, 5))
    
    sns.scatterplot(data=df_gefiltert, x='cursor_x_pix', y='cursor_y_pix', hue='state_marker', palette='bright')
    plt.title("Weg der Maus")
    plt.show()

# Zur Sicherheit plotten wir auch nochmal den Weg der Maus (X und Y)

block_liste = sorted(df_reaching_clean['block'].unique().tolist())
probanden_liste = df_reaching_clean['Participant'].unique().tolist()
trial_liste = sorted(df_reaching_clean['trial'].unique().tolist())
widgets.interact(show_vc, 
                  ausgewaehlter_proband=widgets.Dropdown(options=probanden_liste, description='Participant:'),
                  ausgewaehlter_block=widgets.Dropdown(options=block_liste, description='Block:'),
                  ausgewaehltes_trial=widgets.Dropdown(options=trial_liste, description='Trial:'))



interactive(children=(Dropdown(description='Participant:', options=('CH51', 'IR31', 'LJ25', 'PF13'), value='CH…

<function __main__.show_vc(ausgewaehlter_proband, ausgewaehlter_block, ausgewaehltes_trial)>

Hier bereinigen wir um nur die velocity curve beim reaching anschauen

In [ ]:
def show_vc(ausgewaehlter_proband, ausgewaehlter_block, ausgewaehltes_trial):
    # Das hier ist dein korrekt gefilterter Datensatz
    df_gefiltert = df_reaching_clean[
        (df_reaching_clean['Participant'] == ausgewaehlter_proband) & 
        (df_reaching_clean['block'] == ausgewaehlter_block) &
        (df_reaching_clean['trial'] == ausgewaehltes_trial) &
        (df_reaching_clean ['state_marker'].isin([2, 3, 4]))
    ]
    
    if df_gefiltert.empty:
        print('Keine Daten gefunden')
        return
    
    
    plt.figure(figsize=(10, 5))
    sns.scatterplot(data=df_gefiltert, x='time', y='cursor_velocity', hue='state_marker', palette='bright')
    plt.title("Velocity nach State-Marker")
    plt.show()

    plt.figure(figsize=(10, 5))
    sns.lineplot(data=df_gefiltert, x='time', y='cursor_velocity', hue='state_marker', palette='bright')
    plt.title("Velocity Verlauf (Line Chart)")
    plt.xlabel("Zeit")
    plt.ylabel("Velocity")
    plt.show()
    
    plt.figure(figsize=(5, 5))
    
    sns.scatterplot(data=df_gefiltert, x='cursor_x_pix', y='cursor_y_pix', hue='state_marker', palette='bright')
    plt.title("Weg der Maus")
    plt.show()

# Zur Sicherheit plotten wir auch nochmal den Weg der Maus (X und Y)

block_liste = sorted(df_reaching_clean['block'].unique().tolist())
probanden_liste = df_reaching_clean['Participant'].unique().tolist()
trial_liste = sorted(df_reaching_clean['trial'].unique().tolist())
widgets.interact(show_vc, 
                  ausgewaehlter_proband=widgets.Dropdown(options=probanden_liste, description='Participant:'),
                  ausgewaehlter_block=widgets.Dropdown(options=block_liste, description='Block:'),
                  ausgewaehltes_trial=widgets.Dropdown(options=trial_liste, description='Trial:'))



interactive(children=(Dropdown(description='Participant:', options=('CH51', 'IR31', 'LJ25', 'PF13'), value='CH…

<function __main__.show_vc(ausgewaehlter_proband, ausgewaehlter_block, ausgewaehltes_trial)>

# Continuous Tracking

In [23]:
# 1. Chronologisch ordnen
df_continuous_clean = df_continuous_clean.sort_values(by=['Participant', 'trial_number', 'timestamp'])

# 2. Differenzen berechnen
df_continuous_clean['dt'] = df_continuous_clean.groupby(['Participant', 'trial_number'])['timestamp'].diff()
df_continuous_clean['dx'] = df_continuous_clean.groupby(['Participant', 'trial_number'])['cursor_pos_x'].diff()
df_continuous_clean['dy'] = df_continuous_clean.groupby(['Participant', 'trial_number'])['cursor_pos_y'].diff()

# 3. Pythagoras für Distanz
df_continuous_clean['dist'] = np.sqrt(df_continuous_clean['dx']**2 + df_continuous_clean['dy']**2)

# 4. Geschwindigkeit
df_continuous_clean['cursor_velocity'] = df_continuous_clean['dist'] / df_continuous_clean['dt']

# Kurzer Check
print(df_continuous_clean[['timestamp', 'cursor_pos_x', 'cursor_velocity']].head())

      timestamp  cursor_pos_x  cursor_velocity
4758   0.000008      0.056267              NaN
4759   0.006598      0.056267         0.000000
4760   0.013598      0.056267         2.009618
4761   0.020558      0.056267         2.021184
4762   0.027585      0.042200         2.831132
